# Mistral-7B-Instruct-v0.2 Text Generation — JAX Optimized

This notebook is **Phase 5** of the LLM Response Time Optimizer project.
It validates end-to-end text generation with Mistral-7B-Instruct-v0.2 using
our JAX pipeline and benchmarks it against the PyTorch baseline.

**Model:** `mistralai/Mistral-7B-Instruct-v0.2` (same as PyTorch baseline)

**What this notebook does:**
1. Loads and converts Mistral-7B-Instruct-v0.2 (PyTorch → JAX)
2. Formats prompts with the instruct chat template
3. Runs a single generation test to confirm the pipeline works
4. Benchmarks the same 3 prompts used in the PyTorch baseline
5. Computes tokens/sec, latency, and speedup vs baseline

**PyTorch Baseline Results (from `01_baseline_pytorch.ipynb`):**
| Prompt | PyTorch Latency |
|--------|-----------------|
| Explain quantum computing | 13.69s |
| Write a fibonacci function | 18.08s |
| What is machine learning? | 42.28s |
| **Average** | **24.68s** |

**Requirements:**
- Google Colab with GPU (T4 or better recommended)
- ~14GB RAM for model weights
- Project repo cloned (or uploaded manually)

## 1. Setup Environment

In [1]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# Check for GPU
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL).decode()
    print("GPU detected:")
    for line in gpu_info.split('\n'):
        if 'Tesla' in line or 'T4' in line or 'V100' in line or 'A100' in line or 'RTX' in line:
            print(' ', line.strip())
except Exception:
    print("No GPU detected — generation will be slow on CPU")

Running in Google Colab
GPU detected:
  |   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |


In [2]:
# Install dependencies if needed
!pip install -q torch transformers
!pip install -q jax[cuda12] jaxlib
!pip install -q flax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 60.2 MB/s eta 0:00:00


In [3]:
# Clone the project repo from GitHub
!git clone https://github.com/YashM246/LLM_Response_Time_Optimizer.git

import sys
sys.path.append('./LLM_Response_Time_Optimizer/')

Cloning into 'LLM_Response_Time_Optimizer'...
remote: Enumerating objects: 365, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 365 (delta 67), reused 66 (delta 33), pack-reused 260 (from 1)
Receiving objects: 100% (365/365), 226.94 KiB | 5.40 MiB/s, done.
Resolving deltas: 100% (220/220), done.


In [4]:
import jax
import jax.numpy as jnp
import time
import json

from src.model_conversion import convert_model
from src.cached_generation import generate_text_with_cache, MISTRAL_CONFIG

print(f"JAX version  : {jax.__version__}")
print(f"JAX backend  : {jax.default_backend()}")
print(f"Devices      : {jax.devices()}")
print("Imports OK")

JAX version  : 0.7.2
JAX backend  : gpu
Devices      : [CudaDevice(id=0)]
Imports OK


In [5]:
# Helper: format a raw user prompt into the Mistral instruct chat template.
#
# Mistral-Instruct-v0.2 expects prompts wrapped in [INST]...[/INST] tags.
# Without this, the base model just continues text rather than following
# instructions — which is why the previous run produced C code for fibonacci.
#
# Expected output:
#   <s>[INST] {user_message} [/INST]
#
# We prefer apply_chat_template() but fall back to manual formatting when
# the tokenizer's chat_template attribute is not set (e.g., cached v0.1
# tokenizer or missing template metadata).

def format_prompt(tokenizer, user_message):
    if getattr(tokenizer, 'chat_template', None) is not None:
        messages = [{"role": "user", "content": user_message}]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        # Manual Mistral instruct format — equivalent to apply_chat_template
        return f"<s>[INST] {user_message} [/INST]"

# Will be usable after the model (and tokenizer) is loaded in Section 2.

## 2. Load and Convert Mistral-7B-Instruct-v0.2

Downloads and converts `mistralai/Mistral-7B-Instruct-v0.2` (~14GB).
Same architecture as v0.1 — only the weights differ.

Expected time: **2–5 minutes** on Colab.

In [6]:
print("Loading and converting Mistral-7B-Instruct-v0.2...")
print("(This downloads ~14GB — grab a coffee)\n")

load_start = time.time()

params, tokenizer, model_type = convert_model(model_type="mistral")

load_elapsed = time.time() - load_start
print(f"\nModel loaded and converted in {load_elapsed:.1f}s ({load_elapsed/60:.1f} min)")

Loading and converting Mistral-7B-Instruct-v0.2...
(This downloads ~14GB — grab a coffee)


PyTorch -> JAX Conversion Pipeline (MISTRAL)

[1/4] Loading PyTorch model...
Loading Mistral-7B model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

[OK] Loaded 291 parameters from mistralai/Mistral-7B-Instruct-v0.2
Example weight keys:
    model.embed_tokens.weight: torch.Size([32000, 4096])
    model.layers.0.self_attn.q_proj.weight: torch.Size([4096, 4096])
    model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 4096])
    model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 4096])
    model.layers.0.self_attn.o_proj.weight: torch.Size([4096, 4096])

[2/4] Converting to JAX arrays...
  [OK] Transposed model.layers.0.self_attn.q_proj.weight: torch.Size([4096, 4096]) -> (4096, 4096)
  [OK] Transposed model.layers.0.self_attn.k_proj.weight: torch.Size([1024, 4096]) -> (4096, 1024)
  [OK] Transposed model.layers.0.self_attn.v_proj.weight: torch.Size([1024, 4096]) -> (4096, 1024)
  [OK] Transposed model.layers.0.self_attn.o_proj.weight: torch.Size([4096, 4096]) -> (4096, 4096)
  [OK] Transposed model.layers.0.mlp.gate_proj.weight: torch.Size([14336, 4096]) -> (4096, 14336)
  [OK] Transposed model.layers.0.mlp.up_proj.weigh

## 3. Quick Sanity Check

Verify the converted weights match the expected Mistral-7B architecture.
Fast check — no generation yet.

In [7]:
print("=" * 60)
print("Sanity Checks")
print("=" * 60)

checks_passed = 0
checks_total  = 0

def check(label, condition, detail=""):
    global checks_passed, checks_total
    checks_total += 1
    status = "OK" if condition else "FAIL"
    if condition:
        checks_passed += 1
    suffix = f"  ({detail})" if detail else ""
    print(f"  [{status}] {label}{suffix}")

model_p = params['params']['model']
layer_0  = model_p['layers']['0']

check("Top-level 'model' key present",    'model'        in params['params'])
check("Top-level 'lm_head' key present",  'lm_head'      in params['params'])
check("embed_tokens present",             'embed_tokens'  in model_p)
check("32 transformer layers",            len(model_p['layers']) == 32,
      f"found {len(model_p['layers'])}")
check("Final norm present",               'norm' in model_p)

embed_shape = model_p['embed_tokens']['embedding'].shape
lm_shape    = params['params']['lm_head']['kernel'].shape
norm_shape  = model_p['norm']['kernel'].shape
q_shape     = layer_0['self_attn']['q_proj']['kernel'].shape
k_shape     = layer_0['self_attn']['k_proj']['kernel'].shape

check("Embedding shape (32000, 4096)",  embed_shape == (32000, 4096), str(embed_shape))
check("LM head shape  (4096, 32000)",  lm_shape    == (4096, 32000), str(lm_shape))
check("Final norm shape (4096,)",       norm_shape  == (4096,),       str(norm_shape))
check("Q-proj shape (4096, 4096)",      q_shape     == (4096, 4096),  str(q_shape))
check("K-proj shape (4096, 1024) GQA", k_shape     == (4096, 1024),  str(k_shape))
check("Tokenizer vocab size 32000",     len(tokenizer) == 32000,
      f"found {len(tokenizer)}")

print()
print("=" * 60)
if checks_passed == checks_total:
    print(f"ALL CHECKS PASSED ({checks_passed}/{checks_total})")
else:
    print(f"SOME CHECKS FAILED — {checks_passed}/{checks_total} passed")
print("=" * 60)

Sanity Checks
  [OK] Top-level 'model' key present
  [OK] Top-level 'lm_head' key present
  [OK] embed_tokens present
  [OK] 32 transformer layers  (found 32)
  [OK] Final norm present
  [OK] Embedding shape (32000, 4096)  ((32000, 4096))
  [OK] LM head shape  (4096, 32000)  ((4096, 32000))
  [OK] Final norm shape (4096,)  ((4096,))
  [OK] Q-proj shape (4096, 4096)  ((4096, 4096))
  [OK] K-proj shape (4096, 1024) GQA  ((4096, 1024))
  [OK] Tokenizer vocab size 32000  (found 32000)

ALL CHECKS PASSED (11/11)


## 4. First Generation Test

Run a short generation to confirm the full forward pass works end-to-end.
The prompt is wrapped in the Instruct chat template so the model follows instructions.

The first run will be **slow** — JAX JIT compiles each unique sequence length on first use.

In [8]:
TEST_PROMPT    = "The capital of France is"
MAX_NEW_TOKENS = 20

# Wrap in instruct chat template before passing to the model
formatted_test_prompt = format_prompt(tokenizer, TEST_PROMPT)

print("Running first generation (JIT compilation happens here — will be slow)...")
print(f"Raw prompt      : '{TEST_PROMPT}'")
print(f"Formatted prompt: '{formatted_test_prompt}'")
print(f"Max new tokens  : {MAX_NEW_TOKENS}")
print("-" * 60)

generated_text, stats = generate_text_with_cache(
    params=params,
    tokenizer=tokenizer,
    prompt=formatted_test_prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.0,   # Greedy — deterministic, easiest to verify
    top_k=0,
    use_cache=True,
    model_type="mistral"
)

print("-" * 60)
print(f"\nGenerated text:")
print(f"  {generated_text}")
print(f"\nStats:")
print(f"  Prompt tokens    : {stats['prompt_length']}")
print(f"  Generated tokens : {stats['generated_tokens']}")
print(f"  Time elapsed     : {stats['time_elapsed']:.2f}s")
print(f"  Tokens/sec       : {stats['tokens_per_sec']:.2f}")

Running first generation (JIT compilation happens here — will be slow)...
Raw prompt      : 'The capital of France is'
Formatted prompt: '<s> [INST] The capital of France is [/INST]'
Max new tokens  : 20
------------------------------------------------------------
Prompt: '<s> [INST] The capital of France is [/INST]'
Prompt length: 14 tokens

Prefill phase (processing prompt)...
Mode: Batch prefill (all tokens at once)
[OK] Prefill complete (14 tokens)

Generating 20 new tokens...
  Generated 10/20 tokens...
  Generated 20/20 tokens...
------------------------------------------------------------

Generated text:
  <s><s> [INST] The capital of France is [/INST] The capital city of France is Paris. Paris is one of the most famous cities in the world,

Stats:
  Prompt tokens    : 14
  Generated tokens : 20
  Time elapsed     : 35.85s
  Tokens/sec       : 0.56


## 5. Warmup Run

JAX JIT compiles a separate function for each unique sequence length.
The warmup must generate at least as many tokens as the benchmark run
to pre-compile all shapes — otherwise benchmark times include compilation overhead.

In [9]:
BENCHMARK_TOKENS = 50

print(f"Warming up — generating {BENCHMARK_TOKENS} tokens to pre-compile all shapes...")
print("(This will take a while — subsequent runs will be fast)\n")

warmup_start = time.time()

warmup_prompt = format_prompt(tokenizer, "Hello, how are you?")

_, warmup_stats = generate_text_with_cache(
    params=params,
    tokenizer=tokenizer,
    prompt=warmup_prompt,
    max_new_tokens=BENCHMARK_TOKENS,
    temperature=0.0,
    top_k=0,
    use_cache=True,
    model_type="mistral"
)

warmup_elapsed = time.time() - warmup_start
print(f"Warmup complete in {warmup_elapsed:.1f}s")
print(f"All {BENCHMARK_TOKENS} sequence-length shapes are now compiled.")
print("Benchmark runs below reflect true runtime (no compile overhead).")

Warming up — generating 50 tokens to pre-compile all shapes...
(This will take a while — subsequent runs will be fast)

Prompt: '<s> [INST] Hello, how are you? [/INST]'
Prompt length: 15 tokens

Prefill phase (processing prompt)...
Mode: Batch prefill (all tokens at once)
[OK] Prefill complete (15 tokens)

Generating 50 new tokens...
  Generated 10/50 tokens...
  Generated 20/50 tokens...
  Generated 30/50 tokens...
  Generated 40/50 tokens...
  Generated 50/50 tokens...
Warmup complete in 44.8s
All 50 sequence-length shapes are now compiled.
Benchmark runs below reflect true runtime (no compile overhead).


## 6. Benchmark — 3 Prompts vs PyTorch Baseline

Same 3 prompts as `01_baseline_pytorch.ipynb`, now correctly formatted
with the instruct chat template for a fair comparison.
Both runs use `mistralai/Mistral-7B-Instruct-v0.2`.

In [11]:
# Raw prompts — same as PyTorch baseline
BENCHMARK_PROMPTS = [
    "Explain quantum computing in simple terms",
    "Write a python function to calculate fibonacci",
    "What is machine learning?"
]

# PyTorch baseline latencies (from 01_baseline_pytorch.ipynb)
PYTORCH_LATENCIES = [13.69, 18.08, 42.28]

jax_results = []

print("=" * 70)
print(f"Benchmark: {len(BENCHMARK_PROMPTS)} prompts, {BENCHMARK_TOKENS} tokens each")
print(f"Model: mistralai/Mistral-7B-Instruct-v0.2")
print("=" * 70)

for i, prompt in enumerate(BENCHMARK_PROMPTS):
    # Wrap each prompt in the instruct chat template
    formatted = format_prompt(tokenizer, prompt)

    print(f"\nPrompt {i+1}/{len(BENCHMARK_PROMPTS)}: '{prompt}'")
    print("-" * 70)

    generated_text, stats = generate_text_with_cache(
        params=params,
        tokenizer=tokenizer,
        prompt=formatted,
        max_new_tokens=BENCHMARK_TOKENS,
        temperature=0.7,
        top_k=50,
        use_cache=True,
        model_type="mistral"
    )

    jax_results.append({
        'prompt': prompt,
        'generated_text': generated_text,
        'latency': stats['time_elapsed'],
        'tokens_per_sec': stats['tokens_per_sec'],
        'generated_tokens': stats['generated_tokens']
    })

    print(f"\nOutput: {generated_text[:200]}{'...' if len(generated_text) > 200 else ''}")
    print(f"Latency      : {stats['time_elapsed']:.2f}s")
    print(f"Tokens/sec   : {stats['tokens_per_sec']:.2f}")
    print(f"vs PyTorch   : {PYTORCH_LATENCIES[i]:.2f}s")

print("\n" + "=" * 70)
print("Benchmark complete.")
print("=" * 70)

Benchmark: 3 prompts, 50 tokens each
Model: mistralai/Mistral-7B-Instruct-v0.2

Prompt 1/3: 'Explain quantum computing in simple terms'
----------------------------------------------------------------------
Prompt: '<s> [INST] Explain quantum computing in simple terms [/INST]'
Prompt length: 16 tokens

Prefill phase (processing prompt)...
Mode: Batch prefill (all tokens at once)
[OK] Prefill complete (16 tokens)

Generating 50 new tokens...
  Generated 10/50 tokens...
  Generated 20/50 tokens...
  Generated 30/50 tokens...
  Generated 40/50 tokens...
  Generated 50/50 tokens...

Output: <s><s> [INST] Explain quantum computing in simple terms [/INST] Quantum computing is a new kind of computer technology. While traditional computers use bits, which can only be in one of two states (0 ...
Latency      : 11.12s
Tokens/sec   : 4.50
vs PyTorch   : 13.69s

Prompt 2/3: 'Write a python function to calculate fibonacci'
----------------------------------------------------------------------
Promp

## 7. Results vs PyTorch Baseline

Side-by-side comparison. Both use `mistralai/Mistral-7B-Instruct-v0.2`.

**Important context on the speedup numbers:**

The GPT-2 result (16.32x) compared *our own uncached JAX* against *our own cached JAX* —
eliminating O(n²) recomputation in favour of O(n) KV-cache. That's ~99% of redundant compute gone.

The Mistral comparison below is against PyTorch's `generate()`, which **already uses KV cache
internally**. Both sides are O(n). We are competing on JIT efficiency and XLA optimisation, not
on removing quadratic work. A 2–4x result here is a genuine apples-to-apples win against a
production-grade library.

In [12]:
print("=" * 70)
print("Results: JAX Optimized vs PyTorch Baseline")
print("=" * 70)
print(f"{'Prompt':<42} {'PyTorch':>8} {'JAX':>8} {'Speedup':>8}")
print("-" * 70)

speedups = []

for i, (result, pt_latency) in enumerate(zip(jax_results, PYTORCH_LATENCIES)):
    jax_latency = result['latency']
    speedup     = pt_latency / jax_latency
    speedups.append(speedup)
    prompt_short = result['prompt'][:40] + ('...' if len(result['prompt']) > 40 else '')
    print(f"{prompt_short:<42} {pt_latency:>7.2f}s {jax_latency:>7.2f}s {speedup:>7.2f}x")

print("-" * 70)

avg_pt      = sum(PYTORCH_LATENCIES) / len(PYTORCH_LATENCIES)
avg_jax     = sum(r['latency'] for r in jax_results) / len(jax_results)
avg_speedup = avg_pt / avg_jax
avg_toks    = sum(r['tokens_per_sec'] for r in jax_results) / len(jax_results)

print(f"{'AVERAGE':<42} {avg_pt:>7.2f}s {avg_jax:>7.2f}s {avg_speedup:>7.2f}x")
print("=" * 70)
print(f"\n  Average JAX latency    : {avg_jax:.2f}s")
print(f"  Average PyTorch latency: {avg_pt:.2f}s")
print(f"  Average speedup        : {avg_speedup:.2f}x")
print(f"  Average tokens/sec     : {avg_toks:.2f}")

TARGET_SPEEDUP = 2.5
TARGET_TOKS    = 20.0
print()
print(f"  Target speedup ({TARGET_SPEEDUP}x): {'ACHIEVED' if avg_speedup >= TARGET_SPEEDUP else 'NOT YET'}")
print(f"  Target tok/sec ({TARGET_TOKS}):  {'ACHIEVED' if avg_toks >= TARGET_TOKS else 'NOT YET'}")

Results: JAX Optimized vs PyTorch Baseline
Prompt                                      PyTorch      JAX  Speedup
----------------------------------------------------------------------
Explain quantum computing in simple term...   13.69s   11.12s    1.23x
Write a python function to calculate fib...   18.08s   10.53s    1.72x
What is machine learning?                    42.28s   10.99s    3.85x
----------------------------------------------------------------------
AVERAGE                                      24.68s   10.88s    2.27x

  Average JAX latency    : 10.88s
  Average PyTorch latency: 24.68s
  Average speedup        : 2.27x
  Average tokens/sec     : 4.60

  Target speedup (2.5x): NOT YET
  Target tok/sec (20.0):  NOT YET


## 8. Summary

**What we verified:**
- `Mistral-7B-Instruct-v0.2` loads, converts, and generates correctly via JAX
- Instruct chat template applied — model follows instructions properly
- Batch prefill + KV-Cache + JIT deliver real speedup over PyTorch baseline

**Optimisations applied (in order of impact):**
1. **Batch prefill** — all prompt tokens processed in one parallel forward pass
2. **KV-Cache** — O(N) decode instead of O(N²)
3. **JIT compilation** — XLA-compiled core functions
4. **`jax.lax.scan` decode** — replaces Python for-loop with a single traced XLA program,
   eliminating ~16,000 Python→JAX dispatches per generation call

**On the 16.32x (GPT-2) vs ~2–4x (Mistral) difference:**

These measure different things. GPT-2's 16.32x compared our *own* uncached baseline against our
cached version — quadratic recomputation vs linear KV-cache. Mistral compares our optimised JAX
against PyTorch `generate()`, which already uses KV cache. Both sides are O(N). The Mistral result
is a fair production comparison; the GPT-2 result demonstrates what KV caching alone achieves.

**Next steps:**
1. Re-run this notebook after `lax.scan` changes to measure the scan speedup
2. `04_quality_evaluation.ipynb` — ROUGE-L scores vs PyTorch baseline (target ≥ 98%)
3. `05_results_visualization.ipynb` — comparison plots
4. `06_colab_final_benchmark.ipynb` — 1000-sample Alpaca evaluation